# Módulo 2 — Exploración guiada de modelos de demanda, versión 3

Este notebook no comienza con el modelo más complejo. Construye evidencia por capas:

1. comprender el problema y los datos;
2. establecer baselines difíciles de superar;
3. probar un modelo lineal e inspeccionar sus supuestos;
4. regularizarlo si hay multicolinealidad;
5. probar un árbol para capturar no linealidad;
6. usar ensambles solo si el árbol individual muestra inestabilidad;
7. seleccionar con desempeño, estabilidad, sesgo, interpretabilidad y costo.

**Pregunta analítica:** ¿cuánta energía total se demandará cada día durante el próximo mes?

**Unidad objetivo:** kWh/día. **Horizonte final:** diciembre de 2025, 31 días recursivos. Durante ese horizonte no se usan observaciones reales futuras.

> Cada etapa termina con una conclusión que acepta, condiciona o descarta la alternativa antes de aumentar la complejidad. TC1 y TC2 de enero de 2026 se usan únicamente para cerrar ciclos de facturación de 2025.

## Cómo recorrer el notebook

| Etapa | Pregunta | Alternativa | Puerta para continuar |
|---|---|---|---|
| 0. Referencia | ¿Qué tan difícil es el problema? | Último valor, estacional semanal, media móvil | Ningún modelo se acepta si no mejora una referencia pertinente |
| 1. Relación lineal | ¿Una combinación lineal de rezagos y calendario es suficiente? | Regresión lineal | Revisar VIF, Shapiro–Wilk, Durbin–Watson y residuales |
| 2. Regularización | ¿La colinealidad se controla sin perder interpretabilidad? | Ridge | Comparar sensibilidad al parámetro `alpha` |
| 3. No linealidad | ¿Hay reglas e interacciones que mejoren la predicción? | Árbol de decisión | Medir brecha entrenamiento–validación y profundidad |
| 4. Reducción de varianza | ¿Un ensamble estabiliza al árbol? | Random Forest | Comparar estabilidad por pliegue y costo |
| 5. Aleatoriedad adicional | ¿Cortes más aleatorios aportan evidencia adicional? | Extra Trees | Mantenerlo solo si la mejora justifica menor interpretabilidad |

No se incluye clustering como candidato de pronóstico: es no supervisado y no estima directamente una demanda futura continua. Podría servir después para descubrir segmentos, pero responde otra pregunta.

## Correspondencia con la rúbrica de evaluación

| Criterio | Implementación en esta versión |
|---|---|
| 2.1 Coherencia modelos–requerimientos | Mapa requisito → componente → modelo → métrica → evidencia → brecha |
| 2.2 Evaluación de alternativas | Baselines, modelo estadístico lineal, árbol y dos ensambles; comparación con más de tres criterios |
| 2.3 Supuestos y preparación | Calidad, ADF, VIF, Shapiro–Wilk, Durbin–Watson, Breusch–Pagan y gráficos de diagnóstico; advertencias cuando un supuesto no aplica |
| 2.4 Entrenamiento, validación y calibración | Backtesting cronológico recursivo, sensibilidad de hiperparámetros, brecha de overfitting, curva de aprendizaje e intervalo conformal |
| 2.5 Ajustes iterativos | Diario hipótesis → cambio → evidencia → decisión para cada etapa |
| 2.6 Plan del prototipo | Riesgos, responsables, criterios de terminado, alcance y escenarios de degradación |
| 2.7 Reproducibilidad | Datos consolidados, hash, semilla, versiones, script de preparación, `requirements_colab.txt`, parámetros y exportaciones |

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import scipy
import scipy.stats as stats
import sklearn
import statsmodels
import statsmodels.api as sm
from IPython.display import Markdown, display
from scipy.stats import shapiro

from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

SEED = 42
TARGET_COL = "demanda_total_kwh"
DEVELOPMENT_END = pd.Timestamp("2025-09-30")
CALIBRATION_START = pd.Timestamp("2025-10-01")
CALIBRATION_END = pd.Timestamp("2025-11-30")
OOT_START = pd.Timestamp("2025-12-01")
OOT_END = pd.Timestamp("2025-12-31")
CV_HORIZON_DAYS = 28
N_CV_SPLITS = 3
ALPHA_INTERVAL = 0.10
NOTEBOOK_VERSION = "3.0.0-guiado"

np.random.seed(SEED)
print({
    "python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__,
    "scikit_learn": sklearn.__version__, "scipy": scipy.__version__,
    "statsmodels": statsmodels.__version__, "semilla": SEED,
})

In [ ]:
configured = os.getenv("MODULO2_PROJECT_DIR")
current = Path.cwd().resolve()
candidates = [
    Path(configured).resolve() if configured else None,
    current,
    *current.parents,
    current / "pronostico-demanda-energia",
    Path("/content/pronostico-demanda-energia"),
    Path("/content/drive/MyDrive/Proyecto de Grado/pronostico-demanda-energia"),
]
seen = set()
unique_candidates = []
for candidate in candidates:
    if candidate is not None:
        candidate = candidate.resolve()
        if candidate not in seen:
            seen.add(candidate)
            unique_candidates.append(candidate)

PROJECT_DIR = next(
    (
        candidate
        for candidate in unique_candidates
        if (candidate / "data" / "processed" / "base_modelo_diaria.csv").exists()
    ),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError(
        "No se encontró data/processed. Clone el repositorio, cambie el directorio de trabajo "
        "a su raíz o defina MODULO2_PROJECT_DIR."
    )

DATA_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs"
print(f"Repositorio activo: {PROJECT_DIR}")
print(f"Datos procesados: {DATA_DIR}")

# Parte I — Coherencia entre problema, requerimientos y evidencia

La selección no se basa solamente en “el menor error”. Primero se declara qué requisito cubre cada componente. Los elementos de interfaz, escenarios económicos y alertas que no se resuelven en este notebook se mantienen como brechas explícitas.

In [ ]:
requirements_map = pd.DataFrame([
    ["R1", "Pronóstico y rango", "Modelos + intervalo conformal", "WAPE, cobertura, ancho", "Predicción OOT e intervalo", "Rango aún exploratorio"],
    ["R2", "Comparar con volumen de referencia", "Comparación mensual", "Diferencia absoluta y %", "Tabla real/modelo/referencia", "La interacción corresponde al dashboard"],
    ["R3", "Efecto económico", "No cubierto por el modelo", "Coherencia direccional", "—", "Requiere parámetros comerciales"],
    ["R4", "Baseline y modelo candidato", "Etapas 0–5", "WAPE, MAE, RMSE, sesgo", "Leaderboard temporal", "Revalidar con 2026"],
    ["R5", "Evitar fuga futura", "Split cronológico + recursión", "Cero fechas futuras", "Pruebas automáticas", "Auditar disponibilidad en producción"],
    ["R6", "Error al nivel de decisión", "Evaluación diaria y mensual", "WAPE, MAE, RMSE, sesgo", "Métricas CV/OOT", "Confirmar horizonte con negocio"],
    ["R7", "Incertidumbre", "Conformal separado", "Cobertura y ancho", "Oct–nov calibra; dic prueba", "Menos de dos ciclos"],
    ["R8", "Alertas atípicas", "Fuera del núcleo de pronóstico", "Revisión de alertas", "—", "Módulo posterior"],
    ["R9", "Reproducibilidad", "Hash + semilla + bitácora", "Diferencia <=1e-6", "Repetición automática", "Fijar imagen de ejecución futura"],
    ["R10", "Integrar TC1/TC2", "Preprocesamiento reproducible", "13 periodos y conteos", "reporte_calidad.json", "Actualizar cuando exista demanda real 2026"],
    ["R11", "Calidad y cadencias", "Reglas de calendarización", "Válida, duplicada o inválida", "Reporte de calidad", "Ciclos 2025 cerrados con enero de 2026"],
    ["R12", "Visualización con filtros", "No cubierto por notebook", "Flujos funcionales", "—", "Implementar en prototipo"],
    ["R13", "Exportación", "Carpeta de corrida", "Campos y cifras", "CSV + JSON", "Conectar al frontend"],
    ["R14", "Privacidad", "Datos agregados", "Cero identificadores", "Auditoría de columnas", "Escanear repositorio en CI"],
    ["R15", "Flujo para usuarios", "No cubierto por modelo", "Prueba de usuarios", "—", "Validación del prototipo"],
    ["R16", "Clima/calendario opcional", "Calendario sí; clima no", "Comparación con/sin", "Festivos y ciclos", "Incluir clima solo con evidencia"],
    ["R17", "Sensibilidad económica", "No cubierto", "Casos límite", "—", "Módulo de escenarios"],
    ["R18", "Tiempo razonable", "Cronometría por candidato", "Tiempo de ajuste", "fit_seconds", "Medir también en infraestructura final"],
], columns=["requisito", "necesidad", "componente", "metrica", "evidencia", "brecha_plan_cierre"])
display(requirements_map)

# Parte II — Datos, preparación y supuestos previos

## 1. Carga y controles

La variable objetivo viene de la demanda real horaria agregada a día (`DMRE + DMNR`). TC1/TC2 se usan para calidad, cadencias, segmentación y conciliación. Enero de 2026 completa los ciclos que contienen días del cierre de 2025. Estas variables no se usan como predictores contemporáneos porque la factura se conoce después del consumo.

In [ ]:
base = pd.read_csv(DATA_DIR / "base_modelo_diaria.csv", parse_dates=["fecha"], encoding="utf-8-sig")
monthly_reference = pd.read_csv(
    DATA_DIR / "comparacion_mensual.csv", parse_dates=["periodo"], encoding="utf-8-sig"
)
quality = json.loads((DATA_DIR / "reporte_calidad.json").read_text(encoding="utf-8"))
manifest = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
base = base.sort_values("fecha").set_index("fecha")

expected_dates = pd.date_range("2025-01-01", "2025-12-31", freq="D")
required_columns = {
    TARGET_COL, "demanda_regulada_kwh", "demanda_no_regulada_kwh",
    "consumo_facturado_calendarizado_total_kwh", "cobertura_calendarizacion_completa",
}
assert required_columns.issubset(base.columns)
assert base.index.is_unique and base.index.equals(expected_dates)
assert base[TARGET_COL].notna().all() and base[TARGET_COL].gt(0).all()
assert len(quality["monthly_quality"]) == 13

forbidden_tokens = ["niu", "direccion", "predial", "cedula", "medidor", "latitud", "longitud"]
privacy_findings = [column for column in base.columns if any(token in column.lower() for token in forbidden_tokens)]
assert not privacy_findings and quality["output_checks"]["privacy"]["passed"]
assert abs(quality["output_checks"]["reconciliation_difference_kwh"]) <= 1e-4

display(base.head())
print(f"Filas: {len(base)} | fechas únicas: {base.index.nunique()} | privacidad: APROBADA")

In [ ]:
quality_table = pd.DataFrame([
    {
        "periodo": item["periodo"],
        "filas_tc2": item["tc2"]["rows"],
        "validas": item["tc2"]["valid_rows"],
        "duplicadas": item["tc2"].get("duplicate_rows", 0),
        "invalidas_marcadas": item["tc2"].get("invalid_rows", 0),
        "cruce_tc1_pct": item["tc2"]["tc1_matched_rows"] / max(item["tc2"]["valid_rows"], 1),
        "energia_bruta_kwh": item["tc2"]["energy_raw_kwh"],
        "energia_asignada_2025_kwh": item["tc2"]["energy_allocated_2025_kwh"],
    }
    for item in quality["monthly_quality"]
])
display(quality_table.style.format({"cruce_tc1_pct": "{:.2%}", "energia_bruta_kwh": "{:,.0f}", "energia_asignada_2025_kwh": "{:,.0f}"}))

print("Tratamiento del cierre:", quality["calendarization_rule"]["boundary_effect"])

In [ ]:
closing_quality = next(item for item in quality["monthly_quality"] if item["periodo"] == "2026-01")
closing_energy = closing_quality["tc2"]["energy_allocated_2025_kwh"]
display(Markdown(
    f"### Conclusión de preparación de datos\n\n"
    f"La base se acepta para modelación: contiene 365 fechas, no tiene nulos en la demanda, supera la auditoría de privacidad y concilia sus agregados. "
    f"TC2 de enero de 2026 asigna **{closing_energy:,.0f} kWh** a días de 2025 y corrige el truncamiento del cierre. "
    "Enero de 2026 no se incorpora como demanda real ni como observación de prueba."
))

## 2. Exploración visual y supuestos temporales

Antes de entrenar se revisan cobertura, estacionalidad semanal, valores extremos y relación con la referencia disponible. ADF diagnostica raíz unitaria; no sustituye la inspección visual ni garantiza que exista suficiente historia anual.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
base[TARGET_COL].plot(ax=axes[0, 0], color="#0B6E99", linewidth=1.3)
axes[0, 0].set_title("Demanda total diaria")
axes[0, 0].set_ylabel("kWh")

sns.boxplot(data=base.reset_index(), x="dia_semana", y=TARGET_COL, ax=axes[0, 1], color="#99CDD8")
axes[0, 1].set_title("Distribución por día de semana (0=lunes)")

monthly_reference.set_index("periodo")[["demanda_real_total_kwh", "proyectada_total_kwh"]].plot(
    kind="bar", ax=axes[1, 0], color=["#0B6E99", "#E58B3A"]
)
axes[1, 0].set_title("Demanda real vs. proyección actual")
axes[1, 0].tick_params(axis="x", rotation=45)

base[[TARGET_COL, "consumo_facturado_calendarizado_total_kwh"]].rolling(7).mean().plot(
    ax=axes[1, 1], color=["#0B6E99", "#5B8C5A"]
)
axes[1, 1].axvspan(pd.Timestamp("2025-12-01"), pd.Timestamp("2025-12-31"), color="gray", alpha=0.15)
axes[1, 1].set_title("Demanda vs. TC2 calendarizado — media de 7 días")
plt.tight_layout()
plt.show()

In [ ]:
y = base[TARGET_COL].astype(float).copy()
adf_stat, adf_pvalue, adf_lags, adf_nobs, *_ = adfuller(y, autolag="AIC")
print({"ADF": float(adf_stat), "p_value": float(adf_pvalue), "rezagos": adf_lags, "n": adf_nobs})

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
plot_acf(y, lags=35, ax=axes[0], zero=False)
axes[0].set_title("Autocorrelación de la demanda")
sns.histplot(y, kde=True, ax=axes[1], color="#0B6E99")
axes[1].set_title("Distribución de la variable objetivo")
plt.tight_layout()
plt.show()

display(y.describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]).to_frame("demanda_total_kwh"))

In [ ]:
stationarity_text = "rechaza la hipótesis de raíz unitaria" if adf_pvalue < 0.05 else "no permite rechazar la hipótesis de raíz unitaria"
display(Markdown(
    f"### Conclusión de exploración\n\nLa prueba ADF {stationarity_text} con p={adf_pvalue:.4f}. "
    "La autocorrelación semanal observada justifica conservar rezagos de 7, 14 y 28 días. No se transforma la variable objetivo antes de probar los modelos sencillos."
))

## 3. Ingeniería de variables documentada

Todas las variables se pueden conocer al inicio del horizonte o se construyen exclusivamente con historia anterior:

- rezagos 1, 7, 14 y 28 días;
- media y desviación de los últimos 7 y 28 días;
- cambio respecto a la semana anterior;
- ciclos de día de semana y día del año;
- fin de semana y festivo colombiano;
- tendencia temporal.

No se incluyen TC2 calendarizado, pérdidas observadas del mismo día ni demanda real futura. El pronóstico es recursivo: las predicciones del propio horizonte alimentan los rezagos posteriores.

In [ ]:
LAGS = (1, 7, 14, 28)
COLOMBIA_HOLIDAYS_2025 = pd.to_datetime([
    "2025-01-01", "2025-01-06", "2025-03-24", "2025-04-17", "2025-04-18",
    "2025-05-01", "2025-06-02", "2025-06-23", "2025-06-30", "2025-07-20",
    "2025-08-07", "2025-08-18", "2025-10-13", "2025-11-03", "2025-11-17",
    "2025-12-08", "2025-12-25",
])
FEATURE_COLUMNS = [
    "lag_1", "lag_7", "lag_14", "lag_28", "media_7", "desv_7",
    "media_28", "desv_28", "cambio_7", "dow_sin", "dow_cos",
    "doy_sin", "doy_cos", "es_fin_semana", "es_festivo", "tendencia",
]

def build_feature_row(history: pd.Series, forecast_date: pd.Timestamp) -> dict[str, float]:
    history = history.sort_index()
    if len(history) < max(LAGS):
        raise ValueError("Historia insuficiente para los rezagos.")
    values = history.to_numpy(dtype=float)
    dow, doy = forecast_date.dayofweek, forecast_date.dayofyear
    return {
        "lag_1": values[-1], "lag_7": values[-7], "lag_14": values[-14], "lag_28": values[-28],
        "media_7": values[-7:].mean(), "desv_7": values[-7:].std(ddof=0),
        "media_28": values[-28:].mean(), "desv_28": values[-28:].std(ddof=0),
        "cambio_7": values[-1] - values[-7],
        "dow_sin": np.sin(2 * np.pi * dow / 7), "dow_cos": np.cos(2 * np.pi * dow / 7),
        "doy_sin": np.sin(2 * np.pi * doy / 365.25), "doy_cos": np.cos(2 * np.pi * doy / 365.25),
        "es_fin_semana": float(dow >= 5),
        "es_festivo": float(forecast_date.normalize() in COLOMBIA_HOLIDAYS_2025),
        "tendencia": float((forecast_date - pd.Timestamp("2025-01-01")).days / 365.25),
    }

def make_supervised(series: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
    rows, target, dates = [], [], []
    series = series.sort_index()
    for position in range(max(LAGS), len(series)):
        date = series.index[position]
        rows.append(build_feature_row(series.iloc[:position], date))
        target.append(float(series.iloc[position]))
        dates.append(date)
    X = pd.DataFrame(rows, index=dates)[FEATURE_COLUMNS]
    return X, pd.Series(target, index=dates, name="y")

def recursive_forecast(estimator, history: pd.Series, future_dates: pd.DatetimeIndex) -> pd.Series:
    simulated = history.copy().astype(float)
    predictions = []
    for date in future_dates:
        row = pd.DataFrame([build_feature_row(simulated, date)], columns=FEATURE_COLUMNS)
        prediction = max(0.0, float(estimator.predict(row)[0]))
        predictions.append(prediction)
        simulated.loc[date] = prediction
    return pd.Series(predictions, index=future_dates, name="prediccion")

In [ ]:
display(Markdown(
    "### Conclusión de ingeniería de variables\n\n"
    "Se aprueba un conjunto pequeño de rezagos, estadísticas móviles, tendencia y calendario. "
    "Se excluyen pérdidas del mismo día y TC2 contemporáneo para evitar fuga de información; cualquier aporte futuro de facturación deberá entrar con rezago compatible con su fecha real de disponibilidad."
))

# Parte III — Protocolo común de evaluación

## Métricas elegidas

- **WAPE** es la métrica primaria: expresa el error absoluto total como proporción de la energía real total y se interpreta directamente a nivel del portafolio.
- **MAE** conserva la unidad kWh/día y es menos sensible a extremos.
- **RMSE** penaliza con mayor fuerza errores grandes, relevantes para días de alta desviación.
- **Sesgo** muestra sobrecompra o subestimación sistemática.
- **R²** se conserva como diagnóstico, pero no selecciona el modelo.

Accuracy, precision, recall y F1 no aplican porque el objetivo es continuo. MAPE no es la métrica principal porque pondera cada día por separado y puede volverse inestable cerca de cero; WAPE agrega primero la energía.

In [ ]:
def regression_metrics(actual: pd.Series, predicted: pd.Series) -> dict[str, float]:
    actual, predicted = actual.align(predicted, join="inner")
    error = predicted - actual
    denominator = max(float(actual.abs().sum()), 1e-12)
    return {
        "wape": float(error.abs().sum() / denominator),
        "mae": float(mean_absolute_error(actual, predicted)),
        "rmse": float(mean_squared_error(actual, predicted) ** 0.5),
        "sesgo_pct": float(error.sum() / denominator),
        "r2": float(r2_score(actual, predicted)),
    }

def rolling_splits(series: pd.Series, n_splits: int, horizon: int):
    first_validation = len(series) - n_splits * horizon
    if first_validation < 120:
        raise ValueError("Historia insuficiente para estos pliegues.")
    for split_id in range(n_splits):
        start = first_validation + split_id * horizon
        train, validation = series.iloc[:start], series.iloc[start:start + horizon]
        assert train.index.max() < validation.index.min()
        yield split_id + 1, train, validation

def baseline_forecast(name: str, history: pd.Series, future_dates: pd.DatetimeIndex) -> pd.Series:
    simulated = history.copy().astype(float)
    predictions = []
    for date in future_dates:
        if name == "ultimo_valor":
            value = float(simulated.iloc[-1])
        elif name == "estacional_7":
            value = float(simulated.iloc[-7])
        elif name == "media_movil_7":
            value = float(simulated.iloc[-7:].mean())
        else:
            raise ValueError(name)
        predictions.append(max(0.0, value))
        simulated.loc[date] = value
    return pd.Series(predictions, index=future_dates, name=name)

def build_estimator(family: str, params: dict):
    if family == "regresion_lineal":
        return Pipeline([("scale", StandardScaler()), ("model", LinearRegression())])
    if family == "ridge":
        return Pipeline([("scale", StandardScaler()), ("model", Ridge(**params))])
    if family == "arbol":
        return DecisionTreeRegressor(random_state=SEED, **params)
    if family == "random_forest":
        return RandomForestRegressor(random_state=SEED, n_jobs=-1, **params)
    if family == "extra_trees":
        return ExtraTreesRegressor(random_state=SEED, n_jobs=-1, **params)
    raise ValueError(family)

development = y.loc[:DEVELOPMENT_END]
CV_SPLITS = list(rolling_splits(development, N_CV_SPLITS, CV_HORIZON_DAYS))
experiment_rows, prediction_rows, iteration_journal = [], [], []
model_registry = {}

def evaluate_candidate(stage: str, label: str, family: str, params: dict) -> pd.DataFrame:
    model_registry[label] = {"family": family, "params": params}
    stage_rows = []
    for fold, train, validation in CV_SPLITS:
        X_train, y_train = make_supervised(train)
        estimator = build_estimator(family, params)
        start = time.perf_counter()
        estimator.fit(X_train, y_train)
        fit_seconds = time.perf_counter() - start
        train_pred = pd.Series(estimator.predict(X_train), index=y_train.index)
        train_metrics = regression_metrics(y_train, train_pred)
        validation_pred = recursive_forecast(estimator, train, validation.index)
        validation_metrics = regression_metrics(validation, validation_pred)
        row = {
            "etapa": stage, "label": label, "familia": family,
            "configuracion": json.dumps(params, sort_keys=True), "fold": fold,
            "inicio_validacion": validation.index.min(), "fin_validacion": validation.index.max(),
            "fit_seconds": fit_seconds, "train_mae": train_metrics["mae"],
            "train_wape": train_metrics["wape"], **validation_metrics,
        }
        row["brecha_mae"] = row["mae"] / max(row["train_mae"], 1e-12)
        experiment_rows.append(row)
        stage_rows.append(row)
        prediction_rows.extend({
            "etapa": stage, "label": label, "familia": family, "fold": fold,
            "fecha": date, "real": validation.loc[date], "prediccion": validation_pred.loc[date],
        } for date in validation.index)
    return pd.DataFrame(stage_rows)

def summarize(labels=None) -> pd.DataFrame:
    frame = pd.DataFrame(experiment_rows)
    if labels is not None:
        frame = frame[frame["label"].isin(labels)]
    return (
        frame.groupby(["etapa", "label", "familia", "configuracion"], as_index=False)
        .agg(
            wape_promedio=("wape", "mean"), wape_desv=("wape", "std"),
            mae_promedio=("mae", "mean"), rmse_promedio=("rmse", "mean"),
            sesgo_abs_promedio=("sesgo_pct", lambda values: values.abs().mean()),
            r2_promedio=("r2", "mean"), train_mae_promedio=("train_mae", "mean"),
            brecha_mae_promedio=("brecha_mae", "mean"), fit_seconds=("fit_seconds", "sum"),
        )
        .sort_values(["wape_promedio", "rmse_promedio"])
        .reset_index(drop=True)
    )

In [ ]:
display(Markdown(
    f"### Conclusión del protocolo de evaluación\n\n"
    f"Se aprueban {len(CV_SPLITS)} pliegues cronológicos con pronóstico recursivo de {CV_HORIZON_DAYS} días. "
    "La selección se hará con WAPE promedio y se contrastará con estabilidad, sesgo, sobreajuste, tiempo e interpretabilidad."
))

# Parte IV — Escalamiento gradual de modelos

## Etapa 0 — Baselines

Los baselines no son un trámite. Representan reglas que un analista podría explicar y ejecutar sin entrenamiento. El baseline semanal es especialmente pertinente porque la demanda muestra autocorrelación a siete días.

In [ ]:
BASELINES = ["ultimo_valor", "estacional_7", "media_movil_7"]
baseline_rows = []
for fold, train, validation in CV_SPLITS:
    for baseline in BASELINES:
        pred = baseline_forecast(baseline, train, validation.index)
        metrics = regression_metrics(validation, pred)
        row = {
            "etapa": "0_baseline", "label": baseline, "familia": "baseline",
            "configuracion": baseline, "fold": fold,
            "inicio_validacion": validation.index.min(), "fin_validacion": validation.index.max(),
            "fit_seconds": 0.0, "train_mae": np.nan, "train_wape": np.nan,
            "brecha_mae": np.nan, **metrics,
        }
        experiment_rows.append(row)
        baseline_rows.append(row)
        prediction_rows.extend({
            "etapa": "0_baseline", "label": baseline, "familia": "baseline", "fold": fold,
            "fecha": date, "real": validation.loc[date], "prediccion": pred.loc[date],
        } for date in validation.index)

baseline_summary = summarize(BASELINES)
display(baseline_summary.style.format({
    "wape_promedio": "{:.2%}", "wape_desv": "{:.2%}", "mae_promedio": "{:,.0f}",
    "rmse_promedio": "{:,.0f}", "sesgo_abs_promedio": "{:.2%}", "r2_promedio": "{:.3f}",
}))
best_baseline = baseline_summary.iloc[0]
iteration_journal.append({
    "iteracion": "0 — Baseline",
    "hipotesis": "La demanda semanal reciente contiene una referencia competitiva.",
    "cambio": "Comparar último valor, patrón de 7 días y media móvil.",
    "evidencia": f"Mejor baseline: {best_baseline['label']}; WAPE={best_baseline['wape_promedio']:.2%}.",
    "decision": "Todo modelo posterior debe mejorar esta referencia y justificar su costo.",
})

In [ ]:
display(Markdown(
    f"### Conclusión de baselines\n\nEl baseline **{best_baseline['label']}** fija el umbral con WAPE promedio de **{best_baseline['wape_promedio']:.2%}**. "
    "Ningún modelo entrenado avanzará como candidato si no aporta una mejora verificable frente a esta referencia."
))

## Etapa 1 — Regresión lineal

La regresión lineal es el primer modelo entrenado porque es transparente: permite observar el signo y la magnitud de las relaciones. Su uso exige revisar linealidad aproximada, multicolinealidad, homocedasticidad, normalidad residual —relevante para inferencia— e independencia temporal.

Si esos supuestos fallan, no se ocultan: se documenta si conviene transformar, regularizar o cambiar de familia.

In [ ]:
ols_rows = evaluate_candidate("1_lineal", "regresion_lineal", "regresion_lineal", {})
ols_summary = summarize(["regresion_lineal"])
display(ols_summary.style.format({
    "wape_promedio": "{:.2%}", "wape_desv": "{:.2%}", "mae_promedio": "{:,.0f}",
    "rmse_promedio": "{:,.0f}", "sesgo_abs_promedio": "{:.2%}", "r2_promedio": "{:.3f}",
    "brecha_mae_promedio": "{:.2f}", "fit_seconds": "{:.3f}",
}))

In [ ]:
# Diagnósticos estadísticos sobre el conjunto supervisado de desarrollo.
X_dev, y_dev = make_supervised(development)
X_ols = sm.add_constant(X_dev.astype(float))
ols_diagnostic_model = sm.OLS(y_dev.astype(float), X_ols).fit()
ols_fitted = pd.Series(ols_diagnostic_model.fittedvalues, index=y_dev.index)
ols_residuals = pd.Series(ols_diagnostic_model.resid, index=y_dev.index)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    vif_values = [
        variance_inflation_factor(X_dev.to_numpy(dtype=float), i)
        for i in range(X_dev.shape[1])
    ]
vif_table = pd.DataFrame({
    "variable": FEATURE_COLUMNS,
    "VIF": vif_values,
}).sort_values("VIF", ascending=False)
shapiro_stat, shapiro_p = shapiro(ols_residuals)
dw_stat = float(durbin_watson(ols_residuals))
bp_lm, bp_lm_p, bp_f, bp_f_p = het_breuschpagan(ols_residuals, X_ols)

diagnostic_table = pd.DataFrame([
    ["Shapiro–Wilk", float(shapiro_stat), float(shapiro_p), "p>0.05: no rechazar normalidad"],
    ["Durbin–Watson", dw_stat, np.nan, "aprox. 2: poca autocorrelación de orden 1"],
    ["Breusch–Pagan", float(bp_lm), float(bp_lm_p), "p>0.05: no rechazar homocedasticidad"],
], columns=["prueba", "estadistico", "p_value", "lectura"])
display(diagnostic_table)
display(vif_table.head(12))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].scatter(ols_fitted, ols_residuals, alpha=0.55)
axes[0, 0].axhline(0, color="black")
axes[0, 0].set_title("Residuales vs. ajustados")
axes[0, 0].set_xlabel("Ajustado")
axes[0, 0].set_ylabel("Residual")
stats.probplot(ols_residuals, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title("Q–Q de residuales")
sns.histplot(ols_residuals, kde=True, ax=axes[1, 0], color="#0B6E99")
axes[1, 0].set_title("Distribución residual")
plot_acf(ols_residuals, lags=28, ax=axes[1, 1], zero=False)
axes[1, 1].set_title("ACF residual")
plt.tight_layout()
plt.show()

max_vif = float(vif_table["VIF"].replace([np.inf, -np.inf], np.nan).max())
linear_assumptions_ok = bool(
    shapiro_p > 0.05 and 1.5 <= dw_stat <= 2.5 and bp_lm_p > 0.05 and max_vif < 10
)
print({"max_vif_finito": max_vif, "supuestos_lineales_aprobados_en_conjunto": linear_assumptions_ok})
iteration_journal.append({
    "iteracion": "1 — Regresión lineal",
    "hipotesis": "Una relación lineal de rezagos y calendario puede superar al baseline.",
    "cambio": "Ajustar OLS y auditar VIF, normalidad, homocedasticidad e independencia.",
    "evidencia": (
        f"WAPE={ols_summary.iloc[0]['wape_promedio']:.2%}; VIF máx.={max_vif:.1f}; "
        f"Shapiro p={shapiro_p:.4f}; DW={dw_stat:.2f}; BP p={bp_lm_p:.4f}."
    ),
    "decision": "Probar Ridge para controlar colinealidad; conservar OLS como referencia interpretable.",
})

In [ ]:
linear_wape = float(ols_summary.iloc[0]["wape_promedio"])
linear_reason = []
if max_vif >= 10:
    linear_reason.append(f"multicolinealidad alta, con VIF máximo de {max_vif:,.1f}")
if shapiro_p <= 0.05:
    linear_reason.append(f"residuales no normales según Shapiro–Wilk, p={shapiro_p:.4f}")
if linear_wape >= float(best_baseline["wape_promedio"]):
    linear_reason.append("ausencia de mejora frente al mejor baseline")
display(Markdown(
    f"### Conclusión de regresión lineal\n\nLa regresión lineal obtiene WAPE de **{linear_wape:.2%}**. "
    f"No se promueve como modelo del artefacto por {'; '.join(linear_reason)}. "
    "Se conserva como referencia interpretable y se prueba Ridge para estabilizar los coeficientes correlacionados."
))

## Etapa 2 — Ridge

Ridge mantiene la forma lineal pero penaliza coeficientes grandes. Es una respuesta directa a la multicolinealidad entre rezagos y medias móviles. Se prueban pocos valores de `alpha` para observar sensibilidad; no se lanza todavía una búsqueda masiva.

In [ ]:
RIDGE_ALPHAS = [0.1, 1.0, 10.0, 100.0]
ridge_labels = []
for alpha in RIDGE_ALPHAS:
    label = f"ridge_alpha_{alpha:g}"
    ridge_labels.append(label)
    evaluate_candidate("2_ridge", label, "ridge", {"alpha": alpha})

ridge_summary = summarize(ridge_labels)
display(ridge_summary.style.format({
    "wape_promedio": "{:.2%}", "wape_desv": "{:.2%}", "mae_promedio": "{:,.0f}",
    "rmse_promedio": "{:,.0f}", "sesgo_abs_promedio": "{:.2%}", "r2_promedio": "{:.3f}",
    "brecha_mae_promedio": "{:.2f}", "fit_seconds": "{:.3f}",
}))
best_ridge = ridge_summary.iloc[0]

fig, ax = plt.subplots(figsize=(8, 4))
ridge_plot = ridge_summary.assign(alpha=ridge_summary["label"].str.replace("ridge_alpha_", "").astype(float)).sort_values("alpha")
ax.semilogx(ridge_plot["alpha"], ridge_plot["wape_promedio"], marker="o")
ax.set_title("Sensibilidad de Ridge a alpha")
ax.set_xlabel("alpha")
ax.set_ylabel("WAPE promedio")
ax.yaxis.set_major_formatter(lambda value, _: f"{value:.1%}")
plt.show()

best_ridge_info = model_registry[best_ridge["label"]]
ridge_model = build_estimator(best_ridge_info["family"], best_ridge_info["params"])
ridge_model.fit(X_dev, y_dev)
ridge_coefficients = pd.DataFrame({
    "variable": FEATURE_COLUMNS,
    "coeficiente_estandarizado": ridge_model.named_steps["model"].coef_,
}).assign(magnitud=lambda frame: frame["coeficiente_estandarizado"].abs()).sort_values("magnitud", ascending=False)
display(ridge_coefficients.head(12))

iteration_journal.append({
    "iteracion": "2 — Ridge",
    "hipotesis": "La regularización estabiliza variables correlacionadas sin perder trazabilidad.",
    "cambio": "Comparar cuatro valores de alpha con los mismos pliegues temporales.",
    "evidencia": f"Mejor {best_ridge['label']}; WAPE={best_ridge['wape_promedio']:.2%}; desviación={best_ridge['wape_desv']:.2%}.",
    "decision": "Probar un árbol únicamente para comprobar si relaciones no lineales aportan una mejora material.",
})

In [ ]:
ridge_gain = float(best_baseline["wape_promedio"] - best_ridge["wape_promedio"])
display(Markdown(
    f"### Conclusión de Ridge\n\nLa mejor configuración es **{best_ridge['label']}**, con WAPE de **{best_ridge['wape_promedio']:.2%}**. "
    f"Mejora el baseline en {ridge_gain * 100:.2f} puntos porcentuales y controla la inestabilidad de coeficientes, por lo que queda como candidato lineal. "
    "Se continúa con un árbol para comprobar si la no linealidad produce una mejora material."
))

## Etapa 3 — Árbol de decisión

Un árbol individual permite reglas no lineales e interacciones, y todavía puede explicarse. Su riesgo principal es el sobreajuste: profundidad alta puede memorizar el entrenamiento. Por eso se varían solo `max_depth` y `min_samples_leaf`, y se observa la brecha entre MAE de entrenamiento y validación.

In [ ]:
TREE_DEPTHS = [3, 5, 8, None]
TREE_MIN_LEAVES = [5, 10]
tree_labels = []
for depth in TREE_DEPTHS:
    for min_leaf in TREE_MIN_LEAVES:
        depth_label = "none" if depth is None else str(depth)
        label = f"arbol_d{depth_label}_leaf{min_leaf}"
        tree_labels.append(label)
        evaluate_candidate(
            "3_arbol", label, "arbol",
            {"max_depth": depth, "min_samples_leaf": min_leaf},
        )

tree_summary = summarize(tree_labels)
display(tree_summary.style.format({
    "wape_promedio": "{:.2%}", "wape_desv": "{:.2%}", "mae_promedio": "{:,.0f}",
    "rmse_promedio": "{:,.0f}", "sesgo_abs_promedio": "{:.2%}", "r2_promedio": "{:.3f}",
    "brecha_mae_promedio": "{:.2f}", "fit_seconds": "{:.3f}",
}))
best_tree = tree_summary.iloc[0]

tree_plot = tree_summary.copy()
tree_plot["profundidad"] = tree_plot["label"].str.extract(r"arbol_d([^_]+)")[0]
tree_plot["min_leaf"] = tree_plot["label"].str.extract(r"leaf(\d+)")[0].astype(int)
fig, ax = plt.subplots(figsize=(9, 4))
sns.lineplot(data=tree_plot, x="profundidad", y="wape_promedio", hue="min_leaf", marker="o", ax=ax)
ax.set_title("Sensibilidad del árbol: profundidad y tamaño de hoja")
ax.set_ylabel("WAPE promedio")
ax.yaxis.set_major_formatter(lambda value, _: f"{value:.1%}")
plt.show()

tree_info = model_registry[best_tree["label"]]
tree_model = build_estimator(tree_info["family"], tree_info["params"])
tree_model.fit(X_dev, y_dev)
tree_importance = pd.DataFrame({
    "variable": FEATURE_COLUMNS, "importancia": tree_model.feature_importances_,
}).sort_values("importancia", ascending=False)
display(tree_importance.head(12))

iteration_journal.append({
    "iteracion": "3 — Árbol",
    "hipotesis": "Reglas no lineales pueden capturar interacciones que Ridge no representa.",
    "cambio": "Variar profundidad y mínimo de observaciones por hoja.",
    "evidencia": (
        f"Mejor {best_tree['label']}; WAPE={best_tree['wape_promedio']:.2%}; "
        f"brecha MAE val/train={best_tree['brecha_mae_promedio']:.2f}x."
    ),
    "decision": "Probar Random Forest para reducir la varianza del árbol y contrastar estabilidad.",
})

In [ ]:
tree_gain = float(best_ridge["wape_promedio"] - best_tree["wape_promedio"])
display(Markdown(
    f"### Conclusión del árbol\n\nEl árbol **{best_tree['label']}** alcanza WAPE de **{best_tree['wape_promedio']:.2%}** y mejora Ridge en {tree_gain * 100:.2f} puntos porcentuales. "
    f"La brecha validación/entrenamiento es {best_tree['brecha_mae_promedio']:.2f} veces. "
    "Se mantiene como candidato interpretable, pero se prueba Random Forest para verificar si el promedio de árboles reduce la varianza."
))

## Etapa 4 — Random Forest

Random Forest promedia árboles entrenados con muestras y variables aleatorias. Aumenta costo y reduce interpretabilidad frente al árbol individual, por lo que solo se justifica si mejora desempeño o estabilidad. Se mantiene una grilla pequeña y legible.

In [ ]:
RF_DEPTHS = [6, 10, None]
RF_MIN_LEAVES = [3, 8]
rf_labels = []
for depth in RF_DEPTHS:
    for min_leaf in RF_MIN_LEAVES:
        depth_label = "none" if depth is None else str(depth)
        label = f"rf_d{depth_label}_leaf{min_leaf}"
        rf_labels.append(label)
        evaluate_candidate(
            "4_random_forest", label, "random_forest",
            {"n_estimators": 200, "max_depth": depth, "min_samples_leaf": min_leaf, "max_features": 0.8},
        )

rf_summary = summarize(rf_labels)
display(rf_summary.style.format({
    "wape_promedio": "{:.2%}", "wape_desv": "{:.2%}", "mae_promedio": "{:,.0f}",
    "rmse_promedio": "{:,.0f}", "sesgo_abs_promedio": "{:.2%}", "r2_promedio": "{:.3f}",
    "brecha_mae_promedio": "{:.2f}", "fit_seconds": "{:.3f}",
}))
best_rf = rf_summary.iloc[0]

iteration_journal.append({
    "iteracion": "4 — Random Forest",
    "hipotesis": "Promediar árboles reduce la inestabilidad del árbol individual.",
    "cambio": "Entrenar 200 árboles con tres profundidades y dos tamaños mínimos de hoja.",
    "evidencia": (
        f"Mejor {best_rf['label']}; WAPE={best_rf['wape_promedio']:.2%}; "
        f"desviación={best_rf['wape_desv']:.2%}; tiempo={best_rf['fit_seconds']:.2f}s."
    ),
    "decision": "Probar Extra Trees como última complejidad y exigir mejora medible.",
})

In [ ]:
rf_gain = float(best_tree["wape_promedio"] - best_rf["wape_promedio"])
display(Markdown(
    f"### Conclusión de Random Forest\n\nLa mejor configuración **{best_rf['label']}** obtiene WAPE de **{best_rf['wape_promedio']:.2%}** y mejora el árbol en {rf_gain * 100:.2f} puntos porcentuales. "
    "La mejora justifica conservar el ensamble para la comparación final. Extra Trees se evalúa como último escalón y deberá superar este resultado."
))

## Etapa 5 — Extra Trees

Extra Trees es el último escalón, no el punto de partida. Además de variar muestras y variables, prueba cortes más aleatorios dentro de los árboles. Puede disminuir varianza y entrenar rápido, pero sus decisiones son menos fáciles de explicar globalmente. Se conserva solo si la evidencia supera a Random Forest.

In [ ]:
ET_DEPTHS = [8, None]
ET_MIN_LEAVES = [3, 8, 12]
et_labels = []
for depth in ET_DEPTHS:
    for min_leaf in ET_MIN_LEAVES:
        depth_label = "none" if depth is None else str(depth)
        label = f"et_d{depth_label}_leaf{min_leaf}"
        et_labels.append(label)
        evaluate_candidate(
            "5_extra_trees", label, "extra_trees",
            {"n_estimators": 250, "max_depth": depth, "min_samples_leaf": min_leaf, "max_features": 0.8},
        )

et_summary = summarize(et_labels)
display(et_summary.style.format({
    "wape_promedio": "{:.2%}", "wape_desv": "{:.2%}", "mae_promedio": "{:,.0f}",
    "rmse_promedio": "{:,.0f}", "sesgo_abs_promedio": "{:.2%}", "r2_promedio": "{:.3f}",
    "brecha_mae_promedio": "{:.2f}", "fit_seconds": "{:.3f}",
}))
best_et = et_summary.iloc[0]

iteration_journal.append({
    "iteracion": "5 — Extra Trees",
    "hipotesis": "Cortes más aleatorios reducen varianza sin perder desempeño.",
    "cambio": "Comparar profundidad y tamaño de hoja con 250 árboles.",
    "evidencia": (
        f"Mejor {best_et['label']}; WAPE={best_et['wape_promedio']:.2%}; "
        f"desviación={best_et['wape_desv']:.2%}; tiempo={best_et['fit_seconds']:.2f}s."
    ),
    "decision": "Comparar todas las familias por desempeño, estabilidad, sesgo, overfitting, tiempo e interpretabilidad.",
})

In [ ]:
et_gain = float(best_rf["wape_promedio"] - best_et["wape_promedio"])
display(Markdown(
    f"### Conclusión de Extra Trees\n\nLa configuración **{best_et['label']}** logra WAPE de **{best_et['wape_promedio']:.2%}** y mejora Random Forest en {et_gain * 100:.2f} puntos porcentuales. "
    "Avanza a la selección final por desempeño y estabilidad, con la condición de acompañar el prototipo con una explicación de variables."
))

# Parte V — Comparación sistemática y selección

Para evitar seleccionar por una única cifra, se comparan seis criterios:

1. WAPE promedio;
2. estabilidad entre pliegues;
3. sesgo absoluto;
4. brecha validación/entrenamiento;
5. tiempo de ajuste;
6. interpretabilidad relativa.

La regla principal sigue siendo WAPE. Si dos alternativas quedan a menos de 0,20 puntos porcentuales, se prefiere la más interpretable y estable.

In [ ]:
all_summary = summarize()
best_by_family = (
    all_summary.sort_values(["wape_promedio", "rmse_promedio"])
    .groupby("familia", as_index=False)
    .first()
)
interpretability = {
    "baseline": 5, "regresion_lineal": 5, "ridge": 4,
    "arbol": 4, "random_forest": 2, "extra_trees": 2,
}
complexity = {
    "baseline": 1, "regresion_lineal": 2, "ridge": 2,
    "arbol": 3, "random_forest": 4, "extra_trees": 5,
}
best_by_family["interpretabilidad_1_5"] = best_by_family["familia"].map(interpretability)
best_by_family["complejidad_1_5"] = best_by_family["familia"].map(complexity)
best_by_family["mejora_vs_baseline_pct"] = (
    (best_baseline["wape_promedio"] - best_by_family["wape_promedio"])
    / best_baseline["wape_promedio"]
)
best_by_family = best_by_family.sort_values("wape_promedio").reset_index(drop=True)
display(best_by_family[[
    "familia", "label", "wape_promedio", "wape_desv", "rmse_promedio",
    "sesgo_abs_promedio", "brecha_mae_promedio", "fit_seconds",
    "interpretabilidad_1_5", "complejidad_1_5", "mejora_vs_baseline_pct",
]].style.format({
    "wape_promedio": "{:.2%}", "wape_desv": "{:.2%}", "rmse_promedio": "{:,.0f}",
    "sesgo_abs_promedio": "{:.2%}", "brecha_mae_promedio": "{:.2f}",
    "fit_seconds": "{:.2f}", "mejora_vs_baseline_pct": "{:.1%}",
}))

learned_candidates = best_by_family[best_by_family["familia"].ne("baseline")].copy()
best_wape = learned_candidates["wape_promedio"].min()
eligible = learned_candidates[learned_candidates["wape_promedio"] <= best_wape + 0.002].copy()
selected_row = eligible.sort_values(
    ["interpretabilidad_1_5", "wape_desv", "wape_promedio"],
    ascending=[False, True, True],
).iloc[0]
SELECTED_LABEL = selected_row["label"]
SELECTED_FAMILY = selected_row["familia"]
SELECTED_PARAMS = model_registry[SELECTED_LABEL]["params"]
print("Seleccionado con regla desempeño + parsimonia:", SELECTED_LABEL, SELECTED_PARAMS)

In [ ]:
# Curva de aprendizaje temporal: misma validación final de desarrollo, distinta cantidad de historia.
learning_validation = development.iloc[-CV_HORIZON_DAYS:]
pre_validation = development.iloc[:-CV_HORIZON_DAYS]
window_options = sorted(set([120, 160, 200, len(pre_validation)]))
learning_rows = []
for window in window_options:
    train_subset = pre_validation.iloc[-window:]
    X_subset, y_subset = make_supervised(train_subset)
    estimator = build_estimator(SELECTED_FAMILY, SELECTED_PARAMS)
    estimator.fit(X_subset, y_subset)
    train_pred = pd.Series(estimator.predict(X_subset), index=y_subset.index)
    validation_pred = recursive_forecast(estimator, train_subset, learning_validation.index)
    learning_rows.append({
        "dias_historia": len(train_subset),
        "filas_entrenamiento": len(X_subset),
        "wape_train": regression_metrics(y_subset, train_pred)["wape"],
        "wape_validacion": regression_metrics(learning_validation, validation_pred)["wape"],
    })
learning_curve = pd.DataFrame(learning_rows)
display(learning_curve.style.format({"wape_train": "{:.2%}", "wape_validacion": "{:.2%}"}))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(learning_curve["filas_entrenamiento"], learning_curve["wape_train"], marker="o", label="entrenamiento")
ax.plot(learning_curve["filas_entrenamiento"], learning_curve["wape_validacion"], marker="o", label="validación temporal")
ax.set_title("Curva de aprendizaje temporal del candidato")
ax.set_xlabel("Filas supervisadas de entrenamiento")
ax.set_ylabel("WAPE")
ax.yaxis.set_major_formatter(lambda value, _: f"{value:.1%}")
ax.legend()
plt.show()

In [ ]:
display(Markdown(
    f"### Conclusión de selección\n\nLa regla conjunta selecciona **{SELECTED_LABEL}** de la familia **{SELECTED_FAMILY}**, con WAPE de validación de **{selected_row['wape_promedio']:.2%}**. "
    "La curva de aprendizaje se conserva como evidencia de sensibilidad al volumen histórico; diciembre permanece completamente aislado para la confirmación final."
))

# Parte VI — Calibración separada y prueba final

Los hiperparámetros se eligieron usando únicamente enero–septiembre. Octubre–noviembre se reservan para calibrar el error. Después se reentrena hasta noviembre y se pronostica diciembre. El resultado de diciembre confirma o cuestiona la selección, pero no se usa para volver a elegir.

In [ ]:
calibration_train = y.loc[:DEVELOPMENT_END]
calibration_actual = y.loc[CALIBRATION_START:CALIBRATION_END]
X_cal, y_cal = make_supervised(calibration_train)
calibration_model = build_estimator(SELECTED_FAMILY, SELECTED_PARAMS)
calibration_model.fit(X_cal, y_cal)
calibration_pred = recursive_forecast(calibration_model, calibration_train, calibration_actual.index)
calibration_metrics = regression_metrics(calibration_actual, calibration_pred)

final_train = y.loc[:CALIBRATION_END]
oot_actual = y.loc[OOT_START:OOT_END]
X_final, y_final = make_supervised(final_train)
final_model = build_estimator(SELECTED_FAMILY, SELECTED_PARAMS)
final_model.fit(X_final, y_final)
selected_pred = recursive_forecast(final_model, final_train, oot_actual.index)

oot_predictions = pd.DataFrame({"real": oot_actual, f"modelo_{SELECTED_FAMILY}": selected_pred})
oot_metric_rows = [{"alternativa": f"modelo_{SELECTED_FAMILY}", **regression_metrics(oot_actual, selected_pred)}]
for baseline in BASELINES:
    pred = baseline_forecast(baseline, final_train, oot_actual.index)
    oot_predictions[baseline] = pred
    oot_metric_rows.append({"alternativa": baseline, **regression_metrics(oot_actual, pred)})
oot_metrics = pd.DataFrame(oot_metric_rows).sort_values(["wape", "rmse"]).reset_index(drop=True)

display(pd.DataFrame([calibration_metrics], index=["calibracion_oct_nov"]).style.format({
    "wape": "{:.2%}", "mae": "{:,.0f}", "rmse": "{:,.0f}", "sesgo_pct": "{:.2%}", "r2": "{:.3f}",
}))
display(oot_metrics.style.format({
    "wape": "{:.2%}", "mae": "{:,.0f}", "rmse": "{:,.0f}", "sesgo_pct": "{:.2%}", "r2": "{:.3f}",
}))

In [ ]:
december_reference = monthly_reference.loc[monthly_reference["periodo"].eq(OOT_START)].iloc[0]
monthly_comparison = pd.DataFrame({
    "alternativa": ["real", "proyeccion_referencia", f"modelo_{SELECTED_FAMILY}", *BASELINES],
    "energia_diciembre_kwh": [
        oot_actual.sum(), december_reference["proyectada_total_kwh"], selected_pred.sum(),
        *[oot_predictions[name].sum() for name in BASELINES],
    ],
})
monthly_comparison["error_absoluto_pct"] = (
    (monthly_comparison["energia_diciembre_kwh"] - oot_actual.sum()).abs() / oot_actual.sum()
)
display(monthly_comparison.style.format({"energia_diciembre_kwh": "{:,.0f}", "error_absoluto_pct": "{:.2%}"}))

ax = oot_predictions.plot(figsize=(15, 6), linewidth=1.6)
ax.set_title("Prueba final — diciembre de 2025")
ax.set_ylabel("kWh/día")
plt.show()

In [ ]:
# Intervalo conformal exploratorio: residuales absolutos de octubre–noviembre.
absolute_calibration_error = (calibration_actual - calibration_pred).abs().to_numpy()
n_cal = len(absolute_calibration_error)
quantile_level = min(1.0, np.ceil((n_cal + 1) * (1 - ALPHA_INTERVAL)) / n_cal)
q_error = float(np.quantile(absolute_calibration_error, quantile_level, method="higher"))
interval = pd.DataFrame({
    "real": oot_actual, "prediccion": selected_pred,
    "limite_inferior": np.maximum(0.0, selected_pred - q_error),
    "limite_superior": selected_pred + q_error,
})
coverage = float(((interval["real"] >= interval["limite_inferior"]) &
                  (interval["real"] <= interval["limite_superior"])).mean())
mean_width = float((interval["limite_superior"] - interval["limite_inferior"]).mean())
print({"cobertura_oot": coverage, "ancho_medio_kwh": mean_width, "objetivo_nominal": 1 - ALPHA_INTERVAL})

fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(interval.index, interval["real"], label="real", color="#0B6E99")
ax.plot(interval.index, interval["prediccion"], label="pronóstico", color="#E58B3A")
ax.fill_between(interval.index, interval["limite_inferior"], interval["limite_superior"],
                color="#E58B3A", alpha=0.2, label="rango exploratorio")
ax.set_title("Incertidumbre exploratoria — diciembre")
ax.legend()
plt.show()

In [ ]:
# Interpretabilidad local/global del candidato mediante permutación y residuales OOT.
from sklearn.inspection import permutation_importance

permutation = permutation_importance(
    final_model, X_final.tail(90), y_final.tail(90),
    scoring="neg_mean_absolute_error", n_repeats=20, random_state=SEED, n_jobs=-1,
)
importance_table = pd.DataFrame({
    "variable": FEATURE_COLUMNS,
    "importancia_mae": permutation.importances_mean,
    "desv": permutation.importances_std,
}).sort_values("importancia_mae", ascending=False)
display(importance_table.head(12))

residuals_oot = oot_actual - selected_pred
ljung_box = acorr_ljungbox(residuals_oot, lags=[7, 14], return_df=True)
display(ljung_box)
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].plot(residuals_oot.index, residuals_oot, marker="o")
axes[0].axhline(0, color="black")
axes[0].set_title("Residuales OOT")
sns.histplot(residuals_oot, kde=True, ax=axes[1], color="#0B6E99")
axes[1].set_title("Distribución residual")
plot_acf(residuals_oot, lags=14, ax=axes[2], zero=False)
axes[2].set_title("ACF residual")
plt.tight_layout()
plt.show()

In [ ]:
selected_oot_row = oot_metrics.loc[oot_metrics["alternativa"].eq(f"modelo_{SELECTED_FAMILY}")].iloc[0]
best_oot_baseline = oot_metrics.loc[oot_metrics["alternativa"].isin(BASELINES)].sort_values("wape").iloc[0]
interval_status = "no alcanza" if coverage < 1 - ALPHA_INTERVAL else "alcanza"
display(Markdown(
    f"### Conclusión de prueba final\n\nEl modelo obtiene WAPE de **{selected_oot_row['wape']:.2%}** en diciembre, frente a **{best_oot_baseline['wape']:.2%}** del mejor baseline. "
    f"Se acepta el punto de pronóstico como candidato del prototipo. El intervalo cubre {coverage:.1%} y {interval_status} el objetivo nominal de {1-ALPHA_INTERVAL:.0%}; por ello debe presentarse como exploratorio."
))

# Parte VII — Aprendizajes, plan del prototipo y reproducibilidad

## Diario de iteraciones

El objetivo no es ocultar los modelos que rindieron peor. Cada intento registra la hipótesis, el cambio, la evidencia y la decisión que habilitó —o no— el siguiente nivel de complejidad.

In [ ]:
iteration_table = pd.DataFrame(iteration_journal)
display(iteration_table)

## Plan de implementación del prototipo, priorizado por riesgo

| Prioridad | Riesgo / tarea | Responsable sugerido | Criterio de “terminado” | Respuesta de degradación |
|---|---|---|---|---|
| P0 | Esquema o columnas TC1/TC2 cambian | Datos / calidad | Archivo rechazado con mensaje; cero pérdida silenciosa | Mantener última base versionada y bloquear nueva corrida |
| P0 | Fuga de información en variables | Analítica | Auditoría temporal pasa para todos los pliegues | Desactivar la variable y volver al baseline semanal |
| P0 | Modelo deja de superar baseline | Analítica | WAPE y sesgo monitoreados por periodo | Servir baseline estacional y levantar alerta |
| P1 | Falta TC2 del mes posterior | Datos | Cobertura marcada y conciliación documentada | Ocultar comparaciones incompletas, sin imputar silenciosamente |
| P1 | Deriva de demanda | Analítica / negocio | Umbral de error y sesgo revisado mensualmente | Reentrenar o volver al último modelo aprobado |
| P1 | Tiempo de respuesta >10 s | Backend | Predicción precalculada y filtros agregados | Usar resultados cacheados |
| P2 | Intervalo mal calibrado | Analítica | Cobertura/ancho reportados con más historia | Mostrar rango como exploratorio o retirarlo |
| P2 | Interpretación por usuario | UX / negocio | ≥80% completa el flujo y entiende unidades | Simplificar textos y mostrar baseline junto al modelo |

### Gestión de alcance

El MVP incluye datos, calidad, baselines, un candidato, métricas, pronóstico, rango exploratorio, trazabilidad y exportación. No incluye optimización de compra, oferta real, integración productiva, costos reales, clima obligatorio ni monitoreo continuo. Estas exclusiones evitan convertir un prototipo académico en una promesa comercial.

In [ ]:
# Evidencia final de criterios y pruebas automáticas.
manual_wape = float((oot_actual - selected_pred).abs().sum() / oot_actual.abs().sum())
reported_wape = float(oot_metrics.loc[
    oot_metrics["alternativa"].eq(f"modelo_{SELECTED_FAMILY}"), "wape"
].iloc[0])

repeat_model = build_estimator(SELECTED_FAMILY, SELECTED_PARAMS)
repeat_model.fit(X_final, y_final)
repeat_pred = recursive_forecast(repeat_model, final_train, oot_actual.index)
max_repeat_difference = float((repeat_pred - selected_pred).abs().max())

checks = pd.DataFrame([
    ["2.1 / R1–R18", "Mapa trazable y brechas", len(requirements_map) == 18],
    ["2.2 / R4", "≥4 familias y ≥3 criterios", best_by_family["familia"].nunique() >= 4],
    ["2.3", "ADF, VIF, Shapiro, DW, BP y gráficos", len(vif_table) == len(FEATURE_COLUMNS)],
    ["2.4 / R5", "Pliegues cronológicos", all(train.index.max() < valid.index.min() for _, train, valid in CV_SPLITS)],
    ["2.4 / R6", "Métrica recalculable", abs(manual_wape - reported_wape) <= 1e-12],
    ["2.5", "Iteraciones completas", len(iteration_table) >= 6],
    ["2.7 / R9", "Repetición <=1e-6", max_repeat_difference <= 1e-6],
    ["R10", "13 periodos TC1/TC2", len(quality["monthly_quality"]) == 13],
    ["R11", "Toda fila válida o marcada", all(
        item["tc2"]["valid_rows"] + item["tc2"].get("invalid_rows", 0) + item["tc2"].get("duplicate_rows", 0) == item["tc2"]["rows"]
        for item in quality["monthly_quality"]
    )],
    ["R14", "Sin identificadores", not privacy_findings],
], columns=["criterio", "evidencia", "aprobado"])
display(checks)
assert checks["aprobado"].all()
print("Máxima diferencia al repetir:", max_repeat_difference)

## Documentación necesaria para reproducir

- `Modelo_pronostico_demanda_Modulo2_v3_guiado.ipynb`: experimento completo.
- `Preparacion_datos_pronostico_v3.ipynb`: limpieza, consolidación y exportación.
- `data/processed/base_modelo_diaria.csv`: base final sin identificadores.
- `data/processed/manifest.json`: inventario y hash de las salidas de preparación.
- `data/processed/reporte_calidad.json`: reglas, conteos y conciliaciones.
- `src/preparar_datos_modelo_v3.py`: construcción de la base desde los archivos originales.
- `requirements_colab.txt`: dependencias declaradas.

La adquisición incluye TC1/TC2 de enero de 2025 a enero de 2026; la demanda real permanece en 2025 y las fechas y tamaños quedan en el manifest. El notebook no contiene secretos ni rutas obligatorias fuera de la carpeta del proyecto.

In [ ]:
def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_dir = OUTPUT_DIR / run_id
run_dir.mkdir(parents=True, exist_ok=True)

best_by_family.to_csv(run_dir / "comparacion_familias.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(experiment_rows).to_csv(run_dir / "resultados_por_fold.csv", index=False, encoding="utf-8-sig")
iteration_table.to_csv(run_dir / "diario_iteraciones.csv", index=False, encoding="utf-8-sig")
requirements_map.to_csv(run_dir / "mapa_requisitos.csv", index=False, encoding="utf-8-sig")
learning_curve.to_csv(run_dir / "curva_aprendizaje.csv", index=False, encoding="utf-8-sig")
oot_metrics.to_csv(run_dir / "metricas_oot.csv", index=False, encoding="utf-8-sig")
interval.reset_index(names="fecha").to_csv(run_dir / "predicciones_oot.csv", index=False, encoding="utf-8-sig")

run_log = {
    "run_id": run_id,
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "notebook_version": NOTEBOOK_VERSION,
    "data_file": "data/processed/base_modelo_diaria.csv",
    "data_sha256": file_sha256(DATA_DIR / "base_modelo_diaria.csv"),
    "target": TARGET_COL,
    "seed": SEED,
    "split": {
        "development_end": str(DEVELOPMENT_END.date()),
        "calibration": [str(CALIBRATION_START.date()), str(CALIBRATION_END.date())],
        "out_of_time": [str(OOT_START.date()), str(OOT_END.date())],
        "cv_horizon_days": CV_HORIZON_DAYS,
        "n_cv_splits": N_CV_SPLITS,
    },
    "selected": {"label": SELECTED_LABEL, "family": SELECTED_FAMILY, "params": SELECTED_PARAMS},
    "calibration_metrics": calibration_metrics,
    "oot_metrics": oot_metrics.to_dict(orient="records"),
    "interval": {"alpha": ALPHA_INTERVAL, "coverage": coverage, "mean_width_kwh": mean_width},
    "diagnostics_linear": {
        "adf_p": float(adf_pvalue), "shapiro_p": float(shapiro_p), "durbin_watson": dw_stat,
        "breusch_pagan_p": float(bp_lm_p), "max_finite_vif": max_vif,
    },
    "checks": checks.to_dict(orient="records"),
    "versions": {
        "python": sys.version, "platform": platform.platform(), "pandas": pd.__version__,
        "numpy": np.__version__, "scikit_learn": sklearn.__version__,
        "scipy": scipy.__version__, "statsmodels": statsmodels.__version__,
    },
}
(run_dir / "registro_corrida_guiada.json").write_text(
    json.dumps(run_log, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
)
print(f"Corrida guiada exportada en: {run_dir}")

In [ ]:
selected_oot = oot_metrics.loc[
    oot_metrics["alternativa"].eq(f"modelo_{SELECTED_FAMILY}")
].iloc[0]
decision_text = (
    f"El proceso gradual seleccionó **{SELECTED_LABEL}** ({SELECTED_FAMILY}). "
    f"En diciembre obtuvo WAPE **{selected_oot['wape']:.2%}**, "
    f"RMSE **{selected_oot['rmse']:,.0f} kWh/día** y sesgo **{selected_oot['sesgo_pct']:.2%}**. "
    f"La cobertura del rango exploratorio fue **{coverage:.1%}** frente a un nominal de **{1-ALPHA_INTERVAL:.0%}**. "
    "Por tanto, el punto de pronóstico puede conservarse como candidato académico, pero el intervalo no debe presentarse como calibrado."
)
display(Markdown("# Conclusión del ejercicio guiado\n\n" + decision_text))
display(Markdown(
    "- Ejecutar nuevamente cuando exista demanda real de 2026 y comparar estabilidad anual.\n"
    "- Confirmar con negocio el horizonte real de decisión antes de producción.\n"
    "- Mantener el baseline semanal como mecanismo de degradación.\n"
    "- No añadir clima, costos u otros modelos hasta demostrar que cierran una brecha concreta."
))